# 02 · Visualization: maps from the CRAF'd API

**What this teaches:** how to turn grid-level slices into PRIO-GRID maps: the whole world for one month, one country, several months on a shared scale, the three outcomes side by side, and the recorded deaths next to the forecast.

Grid rows carry each cell's centre as `latitude` and `longitude`. The maps use `views_crafdapi.plotting`. **You need an API key** (see `notebooks/README.md`).

## 1 · Setup

In [ ]:
import os

from dotenv import load_dotenv

import matplotlib.pyplot as plt

from views_crafdapi.client import CrafdApiClient
from views_crafdapi.plotting import add_shared_colorbar, make_subplot_grid, plot_pixel_grid, shared_norm_from_dfs

load_dotenv()  # reads CRAFD_KEY (your key) and, optionally, VIEWS_API_URL

BASE_URL = os.environ.get("VIEWS_API_URL", "https://crafdapi.viewsforecasting.org")
API_KEY = os.environ.get("CRAFD_KEY", "").strip().strip('"')

client = CrafdApiClient(BASE_URL, API_KEY, timeout=300)
print("Base URL:", BASE_URL)
print("API key :", "set" if API_KEY else "MISSING: set it in .env")

In [ ]:
# Stop early, with a clear message, when there is no usable key: otherwise the first data call
# fails with a 401 that does not say the .env is missing.
if not API_KEY or API_KEY == "your-api-key-here":
    raise SystemExit(
        "No usable API key. Copy .env.example to .env at the root of this repository, set\n"
        "CRAFD_KEY to your key (ask VIEWS for one), restart the kernel and\n"
        "run again. 03_offline_demo.ipynb needs no key."
    )

# /health also asks the key store; if only that fails, the data may still be served, so go on.
try:
    health = client.health()
except Exception as e:   # an error answer, a timeout or a dropped connection
    health = {}
    print("Health check unavailable, continuing:", e)
else:
    print("Service:", health["status"], "| forecast loaded:", health.get("forecast_loaded"),
          "| run served:", health.get("served_run"))
if health.get("forecast_loaded") is False:
    raise SystemExit("The service has no forecast loaded yet (it answers 503 to data requests). "
                     "Try again in a few minutes.")

In [ ]:
# The months and Nigeria's GAUL country code, from the small country bulk file (a slice of every
# month of every country would read too much and answer 413: that is what bulk files are for).
countries, MADE_IN = client.download_bulk("country", "sb")
FORECAST_MONTHS = sorted(countries.loc[countries["month"] >= MADE_IN, "month"].unique())
RECORDED_MONTHS = sorted(countries.loc[countries["month"] < MADE_IN, "month"].unique())
FIRST = FORECAST_MONTHS[0]
NIGERIA = int(countries.loc[countries["country_iso3"] == "NGA", "country_code"].iloc[0])
print("First forecast month:", FIRST, "| last recorded month:", RECORDED_MONTHS[-1])

## 2 · The whole world, one month

The expected number of state-based deaths per grid cell in the first forecast month, on a log scale.

In [ ]:
# The whole grid for two months, from the grid bulk file (a whole-grid slice can be too large).
grid_two, _ = client.download_bulk("grid", "sb", [FIRST, RECORDED_MONTHS[-1]],
                            ["month", "priogrid_id", "latitude", "longitude", "country_code", "sb_mean", "sb_actual"])
world = grid_two[grid_two["month"] == FIRST]
# Nigeria's grid cells, asked for by id below: smaller answers than the whole grid.
NGA_CELLS = world.loc[world["country_code"] == NIGERIA, "priogrid_id"].astype(int).tolist()
fig, ax, _ = plot_pixel_grid(world, lon_col="longitude", lat_col="latitude", value_col="sb_mean",
                             transform="log1p", cmap="YlOrRd", figsize=(14, 7))
ax.set_title(f"Expected state-based deaths, {FIRST} (log scale)");

## 3 · One country: expected deaths and the chance of more than 25

The two maps answer different questions: how many deaths to expect, and how likely a serious month is. A chance is between 0 and 1, so it is drawn without the log scale.

In [ ]:
nga = client.slice("grid", "sb", places=NGA_CELLS, months=[FIRST], columns=["sb_mean", "sb_p_gt25"])
fig, axes = make_subplot_grid(2, cols=2, figsize_per_plot=(7, 7))
plot_pixel_grid(nga, ax=axes[0], lon_col="longitude", lat_col="latitude", value_col="sb_mean",
                transform="log1p", cmap="YlOrRd")
axes[0].set_title(f"Nigeria: expected deaths, {FIRST} (log scale)")
plot_pixel_grid(nga, ax=axes[1], lon_col="longitude", lat_col="latitude", value_col="sb_p_gt25",
                transform="none", cmap="Purples")
axes[1].set_title(f"Nigeria: chance of more than 25 deaths, {FIRST}");

## 4 · Several forecast months on one scale

A shared colour scale makes the months comparable: a darker cell is a higher value in every panel.

In [ ]:
months = FORECAST_MONTHS[:3]
panel = client.slice("grid", "sb", places=NGA_CELLS, months=months, columns=["sb_mean"])
by_month = [panel[panel["month"] == m] for m in months]
norm = shared_norm_from_dfs(by_month, value_col="sb_mean", transform="log1p")
fig, axes = make_subplot_grid(len(months), cols=3, figsize_per_plot=(6, 6))
for ax, m, df in zip(axes, months, by_month):
    plot_pixel_grid(df, ax=ax, lon_col="longitude", lat_col="latitude", value_col="sb_mean",
                    transform="log1p", cmap="YlOrRd", norm=norm, colorbar=False)
    ax.set_title(m)
add_shared_colorbar(fig, axes, norm=norm, cmap="YlOrRd", label="expected deaths (log1p)");

## 5 · The three outcomes side by side

State-based (`sb`), non-state (`ns`) and one-sided violence (`os`) are forecast separately; each is its own slice with its own columns. Each panel has its own colour scale here, because the three outcomes differ a lot in size: compare patterns, not shades.

In [ ]:
labels = {"sb": "State-based", "ns": "Non-state", "os": "One-sided"}
fig, axes = make_subplot_grid(3, cols=3, figsize_per_plot=(6, 6))
for ax, (outcome, label) in zip(axes, labels.items()):
    df = client.slice("grid", outcome, places=NGA_CELLS, months=[FIRST], columns=[f"{outcome}_mean"])
    plot_pixel_grid(df, ax=ax, lon_col="longitude", lat_col="latitude", value_col=f"{outcome}_mean",
                    transform="log1p", cmap="YlOrRd")
    ax.set_title(f"{label}, {FIRST}")

## 6 · Recorded deaths next to the forecast: does it look right?

The last recorded month (`sb_actual`, a count) beside the first forecast month (`sb_mean`, an expectation), for the world, Nigeria and Ukraine. The forecast should look like recent conflict: the same places, smoother. A single very bright cell in the middle of a region, where little was recorded, is a warning sign: an earlier data error put deaths at the centres of regions (views-datafactory #547; a cell in Cherkasy, Ukraine, was hit hardest). The two panels have their own colour scales: a recorded count and an expectation differ in kind.

In [ ]:
last = RECORDED_MONTHS[-1]
UKRAINE = int(countries.loc[countries["country_iso3"] == "UKR", "country_code"].iloc[0])
world_recorded = grid_two[grid_two["month"] == last]

def recorded_beside_expected(name, recorded, expected, figsize):
    fig, axes = make_subplot_grid(2, cols=2, figsize_per_plot=figsize)
    plot_pixel_grid(recorded, ax=axes[0], lon_col="longitude", lat_col="latitude", value_col="sb_actual",
                    transform="log1p", cmap="Greys")
    axes[0].set_title(f"{name}: recorded deaths, {last}")
    plot_pixel_grid(expected, ax=axes[1], lon_col="longitude", lat_col="latitude", value_col="sb_mean",
                    transform="log1p", cmap="YlOrRd")
    axes[1].set_title(f"{name}: expected deaths, {FIRST}")

recorded_beside_expected("World", world_recorded, world, (10, 5))
for name, code in (("Nigeria", NIGERIA), ("Ukraine", UKRAINE)):
    keep = lambda df: df[df["country_code"] == code]
    recorded_beside_expected(name, keep(world_recorded), keep(world), (7, 7))

## Next steps

- **`01_quickstart.ipynb`**: the bulk files, slices and the summary columns.
- **`03_offline_demo.ipynb`**: the analytics on synthetic data, no key needed.
- The [data dictionary](../docs/api/data_dictionary.md) explains every column.